# 06 — Focused Experiment Analysis

Reads `results/focused_experiment_results.csv` from `05_classify_focused.py`
and produces:

1. **Efficiency line charts** — Sensitivity / Precision / ROC-AUC vs n_coeffs
2. **Top-20 scatter plot** — Precision vs Sensitivity with article baseline
3. **Chebyshev vs Legendre** — Pairwise Wilcoxon tests
4. **vs OG baseline** — Wilcoxon on repeat-level means
5. **Optimal dimensionality** — smallest n_coeffs without significant loss
6. **Best model summary table**

**Article baseline** (CNN Binary, Youden): Sensitivity = 0.878, Precision = 0.838

In [1]:
# import pandas as pd

# local = pd.read_csv("results/focused_experiment_results.csv")
# hpc = pd.read_csv("results/focused_experiment_results_hpc.csv")

# merged = pd.concat([local, hpc]).drop_duplicates(
#     subset=["representation", "classifier", "split"], keep="first"
# )
# merged.to_csv("results/focused_experiment_results.csv", index=False)
# print(f"Local: {len(local)}, HPC: {len(hpc)}, Merged: {len(merged)}")

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import wilcoxon

ART_SENS = 0.878
ART_PREC = 0.838
ART_ROC = 0.961

EXPERIMENT_DIR = Path.cwd() if Path("results").exists() else Path("transformation_experiment")
RESULTS_DIR = EXPERIMENT_DIR / "results"
MODELS_DIR = EXPERIMENT_DIR / "models_focused"

In [3]:
df = pd.read_csv(RESULTS_DIR / "focused_experiment_results.csv")
print(f"Loaded {len(df)} result rows")
print(f"Representations: {df['representation'].nunique()}")
print(f"Classifiers: {df['classifier'].unique().tolist()}")
print(f"Splits: {df['split'].nunique()}")
df.head()

Loaded 4200 result rows
Representations: 21
Classifiers: ['LogisticRegression', 'SVM_RBF', 'RandomForest', 'XGBoost']
Splits: 50


,threshold,sensitivity,specificity,precision,accuracy,f1,youden_j,roc_auc,pr_auc,brier,log_loss,representation,n_features,classifier,split,best_cv_roc_auc
0,0.482412,0.873874,0.962389,0.850877,0.944938,0.862222,0.836263,0.963406,0.933241,0.057509,0.233924,og_xp_110,110,LogisticRegression,rep0_fold0,0.907849
1,0.381910,0.810811,0.913717,0.697674,0.893428,0.750000,0.724528,0.887348,0.808757,0.080644,0.309569,og_xp_110,110,LogisticRegression,rep0_fold1,0.924947
2,0.492462,0.803571,0.946785,0.789474,0.918295,0.796460,0.750356,0.918950,0.861931,0.071759,0.277967,og_xp_110,110,LogisticRegression,rep0_fold2,0.922964
3,0.587940,0.812500,0.955654,0.819820,0.927176,0.816143,0.768154,0.921128,0.852179,0.075381,0.290866,og_xp_110,110,LogisticRegression,rep0_fold3,0.922132
4,0.407035,0.839286,0.929047,0.746032,0.911190,0.789916,0.768332,0.926750,0.863764,0.076198,0.299655,og_xp_110,110,LogisticRegression,rep0_fold4,0.926167


In [4]:
# Two-level aggregation: fold → repeat → overall
df["repeat"] = df["split"].str.extract(r"rep(\d+)").astype(int)
df["fold"] = df["split"].str.extract(r"fold(\d+)").astype(int)

metric_cols = [
    "roc_auc", "pr_auc", "youden_j", "f1",
    "sensitivity", "specificity", "precision", "accuracy",
    "brier", "log_loss",
]

# Repeat-level means (average across 5 folds within each repeat)
repeat_means = (
    df.groupby(["representation", "n_features", "classifier", "repeat"])[metric_cols]
    .mean()
    .reset_index()
)

# Overall summary: mean ± std across 10 repeats
summary = (
    repeat_means
    .groupby(["representation", "n_features", "classifier"])[metric_cols]
    .agg(["mean", "std"])
)
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]
summary = summary.reset_index().sort_values("f1_mean", ascending=False)

print(f"Repeat-level means: {len(repeat_means)} rows")
print(f"Summary: {len(summary)} configurations")
summary.head(10)

Repeat-level means: 840 rows
Summary: 84 configurations


,representation,n_features,classifier,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,youden_j_mean,youden_j_std,f1_mean,...,specificity_mean,specificity_std,precision_mean,precision_std,accuracy_mean,accuracy_std,brier_mean,brier_std,log_loss_mean,log_loss_std
81,og_xp_110,110,RandomForest,0.941878,0.001376,0.886912,0.002781,0.794786,0.005297,0.828334,...,0.952501,0.004057,0.816331,0.012083,0.930657,0.002077,0.054520,0.000988,0.217514,0.014009
0,chebyshev_10_L2,10,LogisticRegression,0.920265,0.001475,0.861598,0.001632,0.784630,0.004567,0.821496,...,0.951661,0.004218,0.811311,0.012324,0.928135,0.003260,0.072162,0.000422,0.292798,0.001856
83,og_xp_110,110,XGBoost,0.945125,0.002698,0.894605,0.003611,0.787633,0.007391,0.819126,...,0.947673,0.003802,0.801103,0.011443,0.926323,0.002833,0.050562,0.001053,0.205352,0.007730
40,legendre_10_L2,10,LogisticRegression,0.919132,0.002131,0.863568,0.002574,0.781551,0.005348,0.818184,...,0.950198,0.004341,0.806530,0.012939,0.926643,0.003534,0.072278,0.000776,0.291016,0.002796
56,legendre_30_L2,30,LogisticRegression,0.916329,0.002717,0.863611,0.004171,0.776683,0.005853,0.815863,...,0.950689,0.005178,0.807705,0.016345,0.925968,0.003964,0.068976,0.003768,0.270303,0.015505
60,legendre_35_L2,35,LogisticRegression,0.918261,0.002992,0.863832,0.003709,0.778533,0.005660,0.814975,...,0.948784,0.006391,0.802467,0.018348,0.925187,0.003942,0.070831,0.003624,0.276338,0.014136
48,legendre_20_L2,20,LogisticRegression,0.917387,0.002426,0.862177,0.002421,0.784168,0.005065,0.814148,...,0.944927,0.003813,0.791345,0.011143,0.923979,0.002982,0.074623,0.002631,0.293785,0.010059
52,legendre_25_L2,25,LogisticRegression,0.919769,0.001217,0.864900,0.002759,0.780749,0.005073,0.813598,...,0.946346,0.001755,0.794860,0.005725,0.924156,0.001790,0.074050,0.002802,0.290975,0.011015
16,chebyshev_30_L2,30,LogisticRegression,0.918095,0.002869,0.860747,0.002732,0.778399,0.006787,0.813262,...,0.947230,0.003373,0.797471,0.010289,0.924227,0.002659,0.072275,0.003101,0.284559,0.013377
44,legendre_15_L2,15,LogisticRegression,0.917922,0.001848,0.864117,0.002650,0.782145,0.005035,0.812978,...,0.944883,0.004304,0.791242,0.012534,0.923552,0.003298,0.072973,0.001683,0.289436,0.008034


In [5]:
# Parse basis and n_coeffs from representation name
def parse_repr(name):
    if name == "og_xp_110":
        return "og_xp", 110
    parts = name.rsplit("_", 2)
    return parts[0], int(parts[1])

summary[["basis", "n_coeffs"]] = summary["representation"].apply(
    lambda x: pd.Series(parse_repr(x))
)
repeat_means[["basis", "n_coeffs"]] = repeat_means["representation"].apply(
    lambda x: pd.Series(parse_repr(x))
)

# Sort summary by f1_mean descending
summary = summary.sort_values("f1_mean", ascending=False).reset_index(drop=True)
summary.head(10)

,representation,n_features,classifier,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,youden_j_mean,youden_j_std,f1_mean,...,precision_mean,precision_std,accuracy_mean,accuracy_std,brier_mean,brier_std,log_loss_mean,log_loss_std,basis,n_coeffs
0,og_xp_110,110,RandomForest,0.941878,0.001376,0.886912,0.002781,0.794786,0.005297,0.828334,...,0.816331,0.012083,0.930657,0.002077,0.054520,0.000988,0.217514,0.014009,og_xp,110
1,chebyshev_10_L2,10,LogisticRegression,0.920265,0.001475,0.861598,0.001632,0.784630,0.004567,0.821496,...,0.811311,0.012324,0.928135,0.003260,0.072162,0.000422,0.292798,0.001856,chebyshev,10
2,og_xp_110,110,XGBoost,0.945125,0.002698,0.894605,0.003611,0.787633,0.007391,0.819126,...,0.801103,0.011443,0.926323,0.002833,0.050562,0.001053,0.205352,0.007730,og_xp,110
3,legendre_10_L2,10,LogisticRegression,0.919132,0.002131,0.863568,0.002574,0.781551,0.005348,0.818184,...,0.806530,0.012939,0.926643,0.003534,0.072278,0.000776,0.291016,0.002796,legendre,10
4,legendre_30_L2,30,LogisticRegression,0.916329,0.002717,0.863611,0.004171,0.776683,0.005853,0.815863,...,0.807705,0.016345,0.925968,0.003964,0.068976,0.003768,0.270303,0.015505,legendre,30
5,legendre_35_L2,35,LogisticRegression,0.918261,0.002992,0.863832,0.003709,0.778533,0.005660,0.814975,...,0.802467,0.018348,0.925187,0.003942,0.070831,0.003624,0.276338,0.014136,legendre,35
6,legendre_20_L2,20,LogisticRegression,0.917387,0.002426,0.862177,0.002421,0.784168,0.005065,0.814148,...,0.791345,0.011143,0.923979,0.002982,0.074623,0.002631,0.293785,0.010059,legendre,20
7,legendre_25_L2,25,LogisticRegression,0.919769,0.001217,0.864900,0.002759,0.780749,0.005073,0.813598,...,0.794860,0.005725,0.924156,0.001790,0.074050,0.002802,0.290975,0.011015,legendre,25
8,chebyshev_30_L2,30,LogisticRegression,0.918095,0.002869,0.860747,0.002732,0.778399,0.006787,0.813262,...,0.797471,0.010289,0.924227,0.002659,0.072275,0.003101,0.284559,0.013377,chebyshev,30
9,legendre_15_L2,15,LogisticRegression,0.917922,0.001848,0.864117,0.002650,0.782145,0.005035,0.812978,...,0.791242,0.012534,0.923552,0.003298,0.072973,0.001683,0.289436,0.008034,legendre,15


## 1. Efficiency Line Charts

How do Sensitivity, Precision, and ROC-AUC change as we increase the number
of polynomial coefficients? One subplot per classifier.

In [6]:
from scipy import stats as _stats

N_SPLITS = 50  # 10 repeats × 5 folds
_t_crit = _stats.t.ppf(0.975, N_SPLITS - 1)

CLASSIFIERS = summary["classifier"].unique()
BASES = ["chebyshev", "legendre"]
BASE_COLORS = {"chebyshev": "#2196F3", "legendre": "#FF9800"}
BASE_DASH = {"chebyshev": "solid", "legendre": "dash"}

og_summary = summary[summary["basis"] == "og_xp"]

ROW_HEIGHT = 350


def plot_efficiency(metric, ylabel, title_suffix="", art_ref=None):
    n_clf = len(CLASSIFIERS)
    fig = make_subplots(
        rows=n_clf, cols=1,
        subplot_titles=[str(c) for c in CLASSIFIERS],
        shared_xaxes=True,
        vertical_spacing=0.06,
    )

    all_y_vals = []

    for i, clf in enumerate(CLASSIFIERS):
        row = i + 1
        show_legend = (i == 0)

        for basis in BASES:
            mask = (summary["basis"] == basis) & (summary["classifier"] == clf)
            sub = summary[mask].sort_values("n_coeffs")
            if sub.empty:
                continue

            x = sub["n_coeffs"].values
            y_mean = sub[f"{metric}_mean"].values
            y_std = sub[f"{metric}_std"].values
            ci_half = _t_crit * y_std / np.sqrt(N_SPLITS)
            all_y_vals.extend(y_mean + ci_half)
            all_y_vals.extend(y_mean - ci_half)

            fig.add_trace(go.Scatter(
                x=x, y=y_mean,
                mode="lines+markers",
                name=basis.capitalize(),
                legendgroup=basis,
                showlegend=show_legend,
                line=dict(color=BASE_COLORS[basis], dash=BASE_DASH[basis]),
                marker=dict(size=6),
                hovertemplate=f"{basis.capitalize()}<br>"
                              f"n_coeffs=%{{x}}<br>{metric}=%{{y:.4f}}<extra></extra>",
            ), row=row, col=1)

            r, g, b = int(BASE_COLORS[basis][1:3], 16), int(BASE_COLORS[basis][3:5], 16), int(BASE_COLORS[basis][5:7], 16)
            fig.add_trace(go.Scatter(
                x=np.concatenate([x, x[::-1]]),
                y=np.concatenate([y_mean + ci_half, (y_mean - ci_half)[::-1]]),
                fill="toself",
                fillcolor=f"rgba({r},{g},{b},0.15)",
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ), row=row, col=1)

        og_row = og_summary[og_summary["classifier"] == clf]
        if not og_row.empty:
            og_val = og_row[f"{metric}_mean"].values[0]
            all_y_vals.append(og_val)
            fig.add_hline(
                y=og_val, row=row, col=1,
                line=dict(color="#4CAF50", dash="dot", width=1.5),
                annotation_text=f"OG XP 110d ({og_val:.3f})",
                annotation_position="bottom right",
                annotation_font_size=10,
            )

        if art_ref is not None:
            all_y_vals.append(art_ref)
            fig.add_hline(
                y=art_ref, row=row, col=1,
                line=dict(color="#E91E63", dash="dashdot", width=1.5),
                annotation_text=f"Article CNN ({art_ref:.3f})",
                annotation_position="top right",
                annotation_font_size=10,
            )

        fig.update_yaxes(title_text=ylabel, row=row, col=1)

    y_lo, y_hi = min(all_y_vals), max(all_y_vals)
    pad = (y_hi - y_lo) * 0.05
    fig.update_yaxes(range=[y_lo - pad, y_hi + pad])

    fig.update_xaxes(
        title_text="Number of coefficients", row=n_clf, col=1,
        tickvals=list(range(5, 55, 5)),
    )
    fig.update_layout(
        title_text=f"{ylabel} vs Number of Coefficients{title_suffix}",
        height=ROW_HEIGHT * n_clf,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    return fig

In [7]:
fig_sens = plot_efficiency("sensitivity", "Sensitivity", art_ref=ART_SENS)
try:
    fig_sens.write_image(str(RESULTS_DIR / "focused_efficiency_sensitivity.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_sens.show()

Static export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [8]:
fig_prec = plot_efficiency("precision", "Precision", art_ref=ART_PREC)
try:
    fig_prec.write_image(str(RESULTS_DIR / "focused_efficiency_precision.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_prec.show()

Static export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [9]:
fig_auc = plot_efficiency("roc_auc", "ROC-AUC", art_ref=ART_ROC)
try:
    fig_auc.write_image(str(RESULTS_DIR / "focused_efficiency_roc_auc.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_auc.show()

Static export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [10]:
fig_f1 = plot_efficiency("f1", "F1 Score")
try:
    fig_f1.write_image(str(RESULTS_DIR / "focused_efficiency_f1.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_f1.show()

Static export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [11]:
fig_prauc = plot_efficiency("pr_auc", "PR-AUC")
try:
    fig_prauc.write_image(str(RESULTS_DIR / "focused_efficiency_pr_auc.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_prauc.show()

Static export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## 2. Top-20 Scatter Plot: Precision vs Sensitivity

In [12]:
top20 = summary.sort_values("f1_mean", ascending=False).head(10).copy()

CLF_SYMBOLS = {
    "LogisticRegression": "circle",
    "SVM_RBF": "square",
    "RandomForest": "diamond",
    "XGBoost": "triangle-up",
}
BASIS_SCATTER_COLORS = {
    "chebyshev": "#2196F3",
    "legendre": "#FF9800",
    "og_xp": "#4CAF50",
}

fig_scatter = go.Figure()

fig_scatter.add_hline(y=ART_SENS, line=dict(color="#E91E63", dash="dash", width=1), opacity=0.6)
fig_scatter.add_vline(x=ART_PREC, line=dict(color="#E91E63", dash="dash", width=1), opacity=0.6)

fig_scatter.add_trace(go.Scatter(
    x=[ART_PREC], y=[ART_SENS],
    mode="markers+text",
    marker=dict(symbol="star", size=16, color="#E91E63"),
    text=[f"Article CNN ({ART_PREC:.3f}, {ART_SENS:.3f})"],
    textposition="top left",
    textfont=dict(size=10),
    name="Article CNN",
    showlegend=True,
))

shown_clf = set()
shown_basis = set()
for _, row in top20.iterrows():
    basis = row["basis"]
    clf = row["classifier"]
    color = BASIS_SCATTER_COLORS.get(basis, "gray")
    symbol = CLF_SYMBOLS.get(clf, "circle")
    n = int(row["n_coeffs"])
    label = f"{basis[:3].capitalize()}{n}"

    fig_scatter.add_trace(go.Scatter(
        x=[row["precision_mean"]], y=[row["sensitivity_mean"]],
        error_x=dict(type="data", array=[row["precision_std"]], visible=True),
        error_y=dict(type="data", array=[row["sensitivity_std"]], visible=True),
        mode="markers+text",
        marker=dict(symbol=symbol, size=10, color=color, opacity=0.85,
                    line=dict(width=1, color="white")),
        text=[label], textposition="top right", textfont=dict(size=9),
        name=f"{clf} | {basis.capitalize()} {n}",
        legendgroup=clf,
        showlegend=(clf not in shown_clf),
        hovertemplate=(
            f"<b>{clf}</b> — {basis.capitalize()} {n}<br>"
            f"Precision: {row['precision_mean']:.4f} ± {row['precision_std']:.4f}<br>"
            f"Sensitivity: {row['sensitivity_mean']:.4f} ± {row['sensitivity_std']:.4f}<br>"
            f"F1: {row['f1_mean']:.4f}"
            "<extra></extra>"
        ),
    ))
    shown_clf.add(clf)

for basis, color in BASIS_SCATTER_COLORS.items():
    fig_scatter.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=color),
        name=basis.capitalize(),
        showlegend=True,
    ))

fig_scatter.update_layout(
    title="Top 20 Models: Precision vs Sensitivity",
    xaxis_title="Precision (mean ± std)",
    yaxis_title="Sensitivity (mean ± std)",
    height=700, width=900,
    template="plotly_white",
    legend=dict(font_size=10),
)
try:
    fig_scatter.write_image(str(RESULTS_DIR / "focused_top20_scatter.png"), scale=2)
except Exception as e:
    print(f"Static export skipped: {e}")
fig_scatter.show()

Static export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## 3. Chebyshev vs Legendre (Wilcoxon signed-rank)

In [13]:
n_coeffs_grid = sorted(repeat_means[repeat_means["basis"] != "og_xp"]["n_coeffs"].unique())
test_metric = "f1"

comparison_rows = []

for clf in CLASSIFIERS:
    for n in n_coeffs_grid:
        cheb = repeat_means[
            (repeat_means["basis"] == "chebyshev") &
            (repeat_means["n_coeffs"] == n) &
            (repeat_means["classifier"] == clf)
        ].sort_values("repeat")[test_metric].values

        leg = repeat_means[
            (repeat_means["basis"] == "legendre") &
            (repeat_means["n_coeffs"] == n) &
            (repeat_means["classifier"] == clf)
        ].sort_values("repeat")[test_metric].values

        if len(cheb) < 10 or len(leg) < 10:
            continue

        diff = cheb - leg
        try:
            stat, p = wilcoxon(diff, alternative="two-sided")
        except ValueError:
            stat, p = np.nan, 1.0

        comparison_rows.append({
            "classifier": clf,
            "n_coeffs": n,
            "cheb_mean": cheb.mean(),
            "leg_mean": leg.mean(),
            "diff_mean": diff.mean(),
            "p_value": p,
            "significant": p < 0.05,
        })

df_cheb_vs_leg = pd.DataFrame(comparison_rows)

# Holm-Bonferroni correction
from statsmodels.stats.multitest import multipletests
if len(df_cheb_vs_leg) > 0:
    reject, p_adj, _, _ = multipletests(df_cheb_vs_leg["p_value"], method="holm")
    df_cheb_vs_leg["p_adjusted"] = p_adj
    df_cheb_vs_leg["significant_adj"] = reject

print(f"Chebyshev vs Legendre ({test_metric}, Wilcoxon signed-rank, Holm-corrected):")
df_cheb_vs_leg

Chebyshev vs Legendre (f1, Wilcoxon signed-rank, Holm-corrected):


,classifier,n_coeffs,cheb_mean,leg_mean,diff_mean,p_value,significant,p_adjusted,significant_adj
0,RandomForest,5,0.728083,0.768923,-0.040841,0.001953,True,0.078125,False
1,RandomForest,10,0.729145,0.774015,-0.044870,0.001953,True,0.078125,False
2,RandomForest,15,0.700828,0.775558,-0.074730,0.001953,True,0.078125,False
3,RandomForest,20,0.695397,0.775842,-0.080445,0.001953,True,0.078125,False
4,RandomForest,25,0.689924,0.768163,-0.078239,0.001953,True,0.078125,False
5,RandomForest,30,0.683095,0.762826,-0.079731,0.001953,True,0.078125,False
6,RandomForest,35,0.686527,0.758884,-0.072356,0.001953,True,0.078125,False
7,RandomForest,40,0.683045,0.751017,-0.067971,0.001953,True,0.078125,False
8,RandomForest,45,0.679758,0.741836,-0.062078,0.001953,True,0.078125,False
9,RandomForest,50,0.682816,0.724878,-0.042062,0.001953,True,0.078125,False


## 4. Polynomial Representations vs OG Baseline (Wilcoxon)

In [14]:
og_rows = []

for clf in CLASSIFIERS:
    og_scores = repeat_means[
        (repeat_means["basis"] == "og_xp") &
        (repeat_means["classifier"] == clf)
    ].sort_values("repeat")[test_metric].values

    if len(og_scores) < 10:
        continue

    for basis in BASES:
        for n in n_coeffs_grid:
            poly_scores = repeat_means[
                (repeat_means["basis"] == basis) &
                (repeat_means["n_coeffs"] == n) &
                (repeat_means["classifier"] == clf)
            ].sort_values("repeat")[test_metric].values

            if len(poly_scores) < 10:
                continue

            diff = poly_scores - og_scores
            try:
                stat, p = wilcoxon(diff, alternative="two-sided")
            except ValueError:
                stat, p = np.nan, 1.0

            og_rows.append({
                "classifier": clf,
                "basis": basis,
                "n_coeffs": n,
                "poly_mean": poly_scores.mean(),
                "og_mean": og_scores.mean(),
                "diff_mean": diff.mean(),
                "p_value": p,
            })

df_vs_og = pd.DataFrame(og_rows)

if len(df_vs_og) > 0:
    reject, p_adj, _, _ = multipletests(df_vs_og["p_value"], method="holm")
    df_vs_og["p_adjusted"] = p_adj
    df_vs_og["significant_adj"] = reject

print(f"Polynomial vs OG XP 110d ({test_metric}, Holm-corrected):")
print(f"Configurations NOT significantly worse than OG (p_adj > 0.05):")
not_worse = df_vs_og[(~df_vs_og["significant_adj"]) | (df_vs_og["diff_mean"] >= 0)]
not_worse.sort_values(["classifier", "basis", "n_coeffs"])

Polynomial vs OG XP 110d (f1, Holm-corrected):
Configurations NOT significantly worse than OG (p_adj > 0.05):


,classifier,basis,n_coeffs,poly_mean,og_mean,diff_mean,p_value,p_adjusted,significant_adj
20,LogisticRegression,chebyshev,5,0.807147,0.798759,0.008388,0.064453,0.580078,False
21,LogisticRegression,chebyshev,10,0.821496,0.798759,0.022737,0.001953,0.156250,False
22,LogisticRegression,chebyshev,15,0.810247,0.798759,0.011488,0.019531,0.195312,False
23,LogisticRegression,chebyshev,20,0.806304,0.798759,0.007545,0.013672,0.166016,False
24,LogisticRegression,chebyshev,25,0.808889,0.798759,0.010131,0.009766,0.166016,False
...,...,...,...,...,...,...,...,...,...
55,XGBoost,legendre,30,0.775157,0.819126,-0.043969,0.001953,0.156250,False
56,XGBoost,legendre,35,0.772319,0.819126,-0.046807,0.001953,0.156250,False
57,XGBoost,legendre,40,0.772030,0.819126,-0.047096,0.001953,0.156250,False
58,XGBoost,legendre,45,0.758939,0.819126,-0.060187,0.001953,0.156250,False


## 5. Optimal Dimensionality

For each (basis, classifier), find the smallest n_coeffs where adding more
dimensions does **not** significantly improve performance.

In [15]:
optimal_rows = []

for clf in CLASSIFIERS:
    for basis in BASES:
        prev_scores = None
        prev_n = None
        optimal_n = n_coeffs_grid[-1]  # default: largest

        for n in n_coeffs_grid:
            scores = repeat_means[
                (repeat_means["basis"] == basis) &
                (repeat_means["n_coeffs"] == n) &
                (repeat_means["classifier"] == clf)
            ].sort_values("repeat")[test_metric].values

            if prev_scores is not None and len(scores) >= 10:
                diff = scores - prev_scores
                try:
                    _, p = wilcoxon(diff, alternative="greater")
                except ValueError:
                    p = 1.0

                if p > 0.05:
                    optimal_n = prev_n
                    break

            prev_scores = scores
            prev_n = n

        row_data = summary[
            (summary["basis"] == basis) &
            (summary["n_coeffs"] == optimal_n) &
            (summary["classifier"] == clf)
        ]
        if not row_data.empty:
            r = row_data.iloc[0]
            optimal_rows.append({
                "classifier": clf,
                "basis": basis,
                "optimal_n": optimal_n,
                "roc_auc": f"{r['roc_auc_mean']:.4f} ± {r['roc_auc_std']:.4f}",
                "sensitivity": f"{r['sensitivity_mean']:.4f} ± {r['sensitivity_std']:.4f}",
                "precision": f"{r['precision_mean']:.4f} ± {r['precision_std']:.4f}",
                "f1": f"{r['f1_mean']:.4f} ± {r['f1_std']:.4f}",
            })

df_optimal = pd.DataFrame(optimal_rows)
print("Optimal dimensionality (smallest n where adding more doesn't help):")
df_optimal

Optimal dimensionality (smallest n where adding more doesn't help):


,classifier,basis,optimal_n,roc_auc,sensitivity,precision,f1
0,RandomForest,chebyshev,5,0.9085 ± 0.0016,0.7885 ± 0.0115,0.6790 ± 0.0160,0.7281 ± 0.0073
1,RandomForest,legendre,5,0.9123 ± 0.0030,0.8018 ± 0.0073,0.7418 ± 0.0157,0.7689 ± 0.0067
2,LogisticRegression,chebyshev,10,0.9203 ± 0.0015,0.8330 ± 0.0033,0.8113 ± 0.0123,0.8215 ± 0.0064
3,LogisticRegression,legendre,10,0.9191 ± 0.0021,0.8314 ± 0.0031,0.8065 ± 0.0129,0.8182 ± 0.0071
4,XGBoost,chebyshev,5,0.9112 ± 0.0025,0.7918 ± 0.0087,0.7161 ± 0.0178,0.7505 ± 0.0068
5,XGBoost,legendre,5,0.9092 ± 0.0030,0.8052 ± 0.0111,0.7635 ± 0.0136,0.7827 ± 0.0097
6,SVM_RBF,chebyshev,5,0.9274 ± 0.0016,0.8335 ± 0.0049,0.7915 ± 0.0105,0.8112 ± 0.0056
7,SVM_RBF,legendre,5,0.9255 ± 0.0020,0.8331 ± 0.0055,0.7894 ± 0.0157,0.8099 ± 0.0073


## 6. Best Model Summary Table

In [16]:
# Load model manifest for hyperparameters
manifest_path = MODELS_DIR / "model_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
else:
    manifest = {}

# Best per classifier (by F1)
best_rows = []
for clf in CLASSIFIERS:
    clf_data = summary[summary["classifier"] == clf]
    if clf_data.empty:
        continue
    best = clf_data.sort_values("f1_mean", ascending=False).iloc[0]

    # Find a representative model's params from manifest
    repr_name = best["representation"]
    sample_key = f"{repr_name}__{clf}__rep0_fold0.joblib"
    params = manifest.get(sample_key, {}).get("best_params", {})

    beats_sens = "YES" if best["sensitivity_mean"] >= ART_SENS else "no"
    beats_prec = "YES" if best["precision_mean"] >= ART_PREC else "no"

    best_rows.append({
        "Classifier": clf,
        "Representation": repr_name,
        "n_coeffs": int(best["n_coeffs"]),
        "ROC-AUC": f"{best['roc_auc_mean']:.4f} ± {best['roc_auc_std']:.4f}",
        "Sensitivity": f"{best['sensitivity_mean']:.4f} ± {best['sensitivity_std']:.4f}",
        "Precision": f"{best['precision_mean']:.4f} ± {best['precision_std']:.4f}",
        "F1": f"{best['f1_mean']:.4f} ± {best['f1_std']:.4f}",
        "Beats Art. Sens?": beats_sens,
        "Beats Art. Prec?": beats_prec,
        "Best Params": str(params) if params else "—",
    })

df_best = pd.DataFrame(best_rows)
print(f"Article baseline: Sensitivity={ART_SENS}, Precision={ART_PREC}")
print()
df_best

Article baseline: Sensitivity=0.878, Precision=0.838



,Classifier,Representation,n_coeffs,ROC-AUC,Sensitivity,Precision,F1,Beats Art. Sens?,Beats Art. Prec?,Best Params
0,RandomForest,og_xp_110,110,0.9419 ± 0.0014,0.8423 ± 0.0082,0.8163 ± 0.0121,0.8283 ± 0.0036,no,no,"{'n_estimators': 500, 'max_depth': 20, 'min_sa..."
1,LogisticRegression,chebyshev_10_L2,10,0.9203 ± 0.0015,0.8330 ± 0.0033,0.8113 ± 0.0123,0.8215 ± 0.0064,no,no,"{'C': 4.689400963537685, 'class_weight': 'bala..."
2,XGBoost,og_xp_110,110,0.9451 ± 0.0027,0.8400 ± 0.0080,0.8011 ± 0.0114,0.8191 ± 0.0061,no,no,"{'n_estimators': 300, 'max_depth': 10, 'learni..."
3,SVM_RBF,chebyshev_5_L2,5,0.9274 ± 0.0016,0.8335 ± 0.0049,0.7915 ± 0.0105,0.8112 ± 0.0056,no,no,"{'C': 15.670963320055066, 'gamma': 0.019269361..."


## 7. Save Statistical Tests

In [17]:
df_cheb_vs_leg.to_csv(RESULTS_DIR / "focused_cheb_vs_leg.csv", index=False)
df_vs_og.to_csv(RESULTS_DIR / "focused_vs_og.csv", index=False)
df_optimal.to_csv(RESULTS_DIR / "focused_optimal_dims.csv", index=False)
df_best.to_csv(RESULTS_DIR / "focused_best_models.csv", index=False)

print("Saved:")
print("  focused_cheb_vs_leg.csv")
print("  focused_vs_og.csv")
print("  focused_optimal_dims.csv")
print("  focused_best_models.csv")

Saved:
  focused_cheb_vs_leg.csv
  focused_vs_og.csv
  focused_optimal_dims.csv
  focused_best_models.csv


## 9. Performance Gain Relative to Full Model (OG 110-d)

For each polynomial configuration, we compute the relative gain (%) in ROC-AUC
compared to the full 110-dimensional original representation:

$$\text{Gain}(\%) = \frac{\mu(K) - \mu_{\text{OG}}}{\mu_{\text{OG}}} \times 100$$

Negative values indicate performance loss from dimensionality reduction.

In [18]:
GAIN_METRICS = ["roc_auc", "f1"]
GAIN_LABELS = {"roc_auc": "ROC-AUC", "f1": "F1"}

for metric in GAIN_METRICS:
    n_clf = len(CLASSIFIERS)
    fig = make_subplots(
        rows=n_clf, cols=1,
        subplot_titles=[str(c) for c in CLASSIFIERS],
        shared_xaxes=True,
        vertical_spacing=0.08,
    )

    for i, clf in enumerate(CLASSIFIERS):
        row = i + 1
        show_legend = (i == 0)

        og_val = og_summary[og_summary["classifier"] == clf][f"{metric}_mean"].values
        if len(og_val) == 0:
            continue
        og_val = og_val[0]

        for basis in BASES:
            mask = (summary["basis"] == basis) & (summary["classifier"] == clf)
            sub = summary[mask].sort_values("n_coeffs")
            if sub.empty:
                continue

            x = sub["n_coeffs"].values
            y_gain = ((sub[f"{metric}_mean"].values - og_val) / og_val) * 100

            fig.add_trace(go.Scatter(
                x=x, y=y_gain,
                mode="lines+markers",
                name=basis.capitalize(),
                legendgroup=basis,
                showlegend=show_legend,
                line=dict(color=BASE_COLORS[basis], dash=BASE_DASH[basis]),
                marker=dict(size=6),
                hovertemplate=(
                    f"{basis.capitalize()}<br>"
                    f"K=%{{x}}<br>Gain=%{{y:.2f}}%<extra></extra>"
                ),
            ), row=row, col=1)

        fig.add_hline(y=0, line_dash="dot", line_color="rgba(0,0,0,0.3)",
                      annotation_text="OG baseline", row=row, col=1)

    fig.update_layout(
        height=ROW_HEIGHT * n_clf,
        title_text=f"Relative {GAIN_LABELS[metric]} gain vs full OG model (110-d)",
        showlegend=True,
        template="plotly_white",
    )
    fig.update_yaxes(title_text="Gain (%)")
    fig.update_xaxes(title_text="Number of coefficients", row=n_clf, col=1)
    fig.show()

## 10. 1-SE Near-Optimal Regions

Following the 1-SE (one standard error) rule, we identify the smallest number of
coefficients that achieves performance statistically compatible with the best
observed configuration. For maximization metrics:

$$\mu(K) \geq \mu(K_{\text{best}}) - \text{SE}_{\text{best}}$$

where $\text{SE}_{\text{best}} = \sigma(K_{\text{best}}) / \sqrt{R}$, with $R$ repeats.

Any $K$ satisfying this inequality is **near-optimal**. The smallest such $K$ is
the parsimonious 1-SE choice.

In [19]:
R = repeat_means["repeat"].nunique()
SE_METRIC = "roc_auc"

one_se_rows = []
one_se_summary = []

for clf in CLASSIFIERS:
    for basis in BASES:
        mask = (summary["basis"] == basis) & (summary["classifier"] == clf)
        sub = summary[mask].sort_values("n_coeffs")
        if sub.empty:
            continue

        best_idx = sub[f"{SE_METRIC}_mean"].idxmax()
        best_row = sub.loc[best_idx]
        mu_best = best_row[f"{SE_METRIC}_mean"]
        se_best = best_row[f"{SE_METRIC}_std"] / np.sqrt(R)
        k_best = int(best_row["n_coeffs"])
        threshold = mu_best - se_best

        near_optimal_ks = sub[sub[f"{SE_METRIC}_mean"] >= threshold]["n_coeffs"].values
        k_1se = int(near_optimal_ks.min()) if len(near_optimal_ks) > 0 else k_best

        one_se_summary.append({
            "classifier": clf,
            "basis": basis,
            "K_best": k_best,
            "K_1se": k_1se,
            "mu_best": mu_best,
            "se_best": se_best,
            "threshold": threshold,
            "reduction": f"{110 // k_1se}x (110 → {k_1se})",
        })

        for _, r in sub.iterrows():
            one_se_rows.append({
                "classifier": clf,
                "basis": basis,
                "n_coeffs": int(r["n_coeffs"]),
                "near_optimal": r[f"{SE_METRIC}_mean"] >= threshold,
            })

df_1se = pd.DataFrame(one_se_rows)
df_1se_summary = pd.DataFrame(one_se_summary)

print(f"1-SE analysis ({SE_METRIC.upper()}, R={R} repeats):\n")
df_1se_summary[["classifier", "basis", "K_best", "K_1se",
                 "mu_best", "threshold", "reduction"]]

1-SE analysis (ROC_AUC, R=10 repeats):



,classifier,basis,K_best,K_1se,mu_best,threshold,reduction
0,RandomForest,chebyshev,5,5,0.908547,0.908026,22x (110 → 5)
1,RandomForest,legendre,40,5,0.912562,0.911672,22x (110 → 5)
2,LogisticRegression,chebyshev,10,10,0.920265,0.919799,11x (110 → 10)
3,LogisticRegression,legendre,25,25,0.919769,0.919384,4x (110 → 25)
4,XGBoost,chebyshev,30,10,0.912913,0.912297,11x (110 → 10)
5,XGBoost,legendre,40,40,0.919109,0.918498,2x (110 → 40)
6,SVM_RBF,chebyshev,5,5,0.927371,0.926861,22x (110 → 5)
7,SVM_RBF,legendre,5,5,0.925523,0.924879,22x (110 → 5)


In [20]:
CLF_ORDER = ["SVM_RBF", "RandomForest", "LogisticRegression", "XGBoost"]
CLF_DISPLAY = {
    "SVM_RBF": "SVM",
    "RandomForest": "Random Forest",
    "LogisticRegression": "Logistic Regression",
    "XGBoost": "XGBoost",
}
CLF_REGION_COLORS = {
    "SVM_RBF": "#E53935",
    "RandomForest": "#43A047",
    "LogisticRegression": "#FB8C00",
    "XGBoost": "#1E88E5",
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"<b>{SE_METRIC.upper()}</b> — Chebyshev stable-optimal region",
        f"<b>{SE_METRIC.upper()}</b> — Legendre stable-optimal region",
    ],
    shared_yaxes=True,
    horizontal_spacing=0.28,
)

ANNOT_X = 56

for col_idx, basis in enumerate(["chebyshev", "legendre"], 1):
    for clf in CLF_ORDER:
        y_label = CLF_DISPLAY[clf]
        color = CLF_REGION_COLORS[clf]

        mask_s = (
            (df_1se_summary["classifier"] == clf)
            & (df_1se_summary["basis"] == basis)
        )
        if not mask_s.any():
            continue
        info = df_1se_summary[mask_s].iloc[0]
        k_best = int(info["K_best"])
        k_1se = int(info["K_1se"])

        near_mask = (
            (df_1se["classifier"] == clf)
            & (df_1se["basis"] == basis)
            & df_1se["near_optimal"]
        )
        near_ks = sorted(df_1se[near_mask]["n_coeffs"].values.astype(int))
        if not near_ks:
            near_ks = [k_1se]
        k_min, k_max = min(near_ks), max(near_ks)

        # ── stable region bar ──
        fig.add_trace(
            go.Scatter(
                x=[k_min, k_max], y=[y_label, y_label],
                mode="lines",
                line=dict(color=color, width=14),
                showlegend=False, hoverinfo="skip",
            ),
            row=1, col=col_idx,
        )

        # ── dotted connector when 1-SE choice is outside the bar ──
        if k_1se < k_min:
            fig.add_trace(
                go.Scatter(
                    x=[k_1se, k_min], y=[y_label, y_label],
                    mode="lines",
                    line=dict(color=color, width=2, dash="dot"),
                    showlegend=False, hoverinfo="skip",
                ),
                row=1, col=col_idx,
            )

        # ── best-K marker (filled circle) ──
        fig.add_trace(
            go.Scatter(
                x=[k_best], y=[y_label], mode="markers",
                marker=dict(color="#333", size=11, symbol="circle"),
                showlegend=False,
                hovertemplate=f"Best K = {k_best}<extra></extra>",
            ),
            row=1, col=col_idx,
        )

        # ── 1-SE choice marker (open diamond) ──
        fig.add_trace(
            go.Scatter(
                x=[k_1se], y=[y_label], mode="markers",
                marker=dict(
                    color="white", size=13, symbol="diamond",
                    line=dict(width=2.5, color="#333"),
                ),
                showlegend=False,
                hovertemplate=f"1-SE K = {k_1se}<extra></extra>",
            ),
            row=1, col=col_idx,
        )

        # ── range label below bar ──
        if k_min != k_max:
            fig.add_annotation(
                x=(k_min + k_max) / 2, y=y_label,
                yshift=-18,
                text=f"<span style='color:{color}'>K {k_min}–{k_max}</span>",
                showarrow=False, font=dict(size=10),
                row=1, col=col_idx,
            )

        # ── right-side stats annotation ──
        pct_fewer = (1 - k_1se / 110) * 100

        k1se_perf = summary[
            (summary["basis"] == basis)
            & (summary["n_coeffs"] == k_1se)
            & (summary["classifier"] == clf)
        ]
        roc_1se = (
            k1se_perf[f"{SE_METRIC}_mean"].values[0]
            if not k1se_perf.empty else info["mu_best"]
        )
        og_roc = og_summary[
            og_summary["classifier"] == clf
        ][f"{SE_METRIC}_mean"].values
        delta_pp = (roc_1se - og_roc[0]) if len(og_roc) > 0 else 0
        direction = "better" if delta_pp >= 0 else "lower"

        fig.add_annotation(
            x=ANNOT_X, y=y_label,
            text=(
                f"<b>K = {k_1se}</b><br>"
                f"{pct_fewer:.1f}% fewer coeff.<br>"
                f"{abs(delta_pp):.3f} {direction}"
            ),
            showarrow=False, xanchor="left", font=dict(size=11),
            row=1, col=col_idx,
        )

# ── legend-only traces ──
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(color="#333", size=10, symbol="circle"),
    name="Best performing K",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(color="white", size=13, symbol="diamond",
                line=dict(width=2, color="#333")),
    name="Earliest 1-SE-sufficient K",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines",
    line=dict(color="#9E9E9E", width=10),
    name="Stable 1-SE region containing best K",
), row=1, col=1)

fig.update_layout(
    title_text=(
        "<b>Near-optimal coefficient regions under 1-SE criterion</b>"
    ),
    height=380, width=1100,
    template="plotly_white",
    legend=dict(
        orientation="h", yanchor="top", y=-0.18,
        xanchor="center", x=0.5,
    ),
    margin=dict(r=30, l=140, t=80, b=80),
)
fig.update_xaxes(
    title_text="Retained coefficient count K",
    range=[0, 58], dtick=10,
)
fig.update_yaxes(
    categoryorder="array",
    categoryarray=list(reversed([CLF_DISPLAY[c] for c in CLF_ORDER])),
)
fig.show()

## Key Takeaways

1. **Dimensionality reduction works.** Under the 1-SE criterion, SVM and
   RandomForest reach near-optimal ROC-AUC at just **K = 5** coefficients
   (22× reduction from 110-d). Logistic Regression needs K = 10 for Chebyshev
   (11× reduction). XGBoost with Chebyshev also saturates at K = 10.

2. **Chebyshev ≈ Legendre.** After Holm–Bonferroni correction, no significant
   F1 difference between polynomial bases for LogReg, XGBoost, or SVM.
   RandomForest favours Legendre (higher F1 across all K), but the effect
   does not survive multiple-testing correction.

3. **Compact polynomials rival the full model.** SVM with 5 Chebyshev
   coefficients achieves ROC-AUC = 0.927 vs the OG 110-d SVM's 0.939
   (only ~1.3 % loss). LogReg with 10 Chebyshev coefficients *improves*
   F1 over OG LogReg (0.821 vs 0.799). No polynomial configuration is
   significantly worse than OG after Holm correction.

4. **Gap to the Article CNN remains.** The article's CNN achieves
   Sensitivity = 0.878, Precision = 0.838, ROC-AUC = 0.961.
   Our best models reach ~0.84 sensitivity and ~0.82 precision — close,
   but not matching the deep-learning baseline. The advantage is
   interpretability and extreme dimensionality reduction (5–10 features
   instead of 110).